# JackSparrow v43 — Delta Exchange India (system-aligned Colab)

Train and export a **v43** bundle (`metadata_v43.json` + `model_artifact_v43.pkl`) compatible with
`JackSparrowV43Node` in this repository.

This is **not** the scratch baseline in `notebooks/delta_india_training_colab.ipynb` — it follows
`V43_CANONICAL_FEATURES`, `training_forward_bars` = 120, and the v43 artifact layout.


## References (in your cloned repo)

- `docs/03-ml-models.md` — model storage and v43 path
- `docs/feature_entrypoints_audit.md` — train/serve entrypoints
- `feature_store/jacksparrow_v43_contract.py` — feature order + version gate
- `feature_store/jacksparrow_v43_build_matrix.py` — `build_v43_feature_matrix`
- `agent/models/v43_pickle_shims.py` — `EnsembleModel`, `LGBMModel`, `FeatureEngineer` pickle contract


In [21]:
# Minimal Colab dependencies (pin versions in prod if needed)
%pip install -q pandas numpy requests joblib scikit-learn lightgbm xgboost


## 1) Repository on `sys.path`

**Preferred:** just run the next cell — it clones the **`major-rework`** branch automatically and switches to it if an older clone is already present.

Manual equivalent:

```bash
git clone --branch major-rework https://github.com/energyforreal/JackSparrow.git /content/trading-agent
```

If you omit the target directory, Git creates `/content/JackSparrow` — the next cell detects that too.

(Private repo: use a [personal access token](https://github.com/settings/tokens) or SSH URL instead.)

This notebook needs **`feature_store/jacksparrow_v43_build_matrix.py`** and **`jacksparrow_v43_contract.py`** on the branch you clone. If the repo-root cell fails after `git pull`, push those files from your local checkout.

Alternatively mount Google Drive containing a checkout and set environment variable `TRADING_AGENT_ROOT`
to that folder before importing.


In [ ]:
import subprocess
from pathlib import Path

_CLONE = Path("/content/trading-agent")
_URL = "https://github.com/energyforreal/JackSparrow.git"
_BRANCH = "major-rework"

if (_CLONE / ".git").is_dir():
    print("Repo already present:", _CLONE)
    result = subprocess.run(
        ["git", "-C", str(_CLONE), "rev-parse", "--abbrev-ref", "HEAD"],
        capture_output=True, text=True,
    )
    current_branch = result.stdout.strip()
    if current_branch != _BRANCH:
        print(f"Switching from '{current_branch}' to '{_BRANCH}'...")
        subprocess.run(["git", "-C", str(_CLONE), "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", str(_CLONE), "checkout", _BRANCH], check=True)
        subprocess.run(["git", "-C", str(_CLONE), "pull", "origin", _BRANCH], check=True)
    else:
        subprocess.run(["git", "-C", str(_CLONE), "pull", "origin", _BRANCH], check=True)
    print(f"On branch '{_BRANCH}', up to date.")
else:
    subprocess.run(["git", "clone", "--branch", _BRANCH, _URL, str(_CLONE)], check=True)
    print(f"Cloned branch '{_BRANCH}' to", _CLONE)


## 1b) If the repo-root cell fails: wrong branch cloned

Diagnostics showing **`/content/trading-agent/.../jacksparrow_v43_build_matrix.py` exists: False** mean Colab cloned the **wrong branch** — the v43 modules only exist on `major-rework`, not `main`.

**Fix A — re-run the §1 clone cell (handles this automatically).**

The §1 clone cell now always clones / switches to `major-rework`. Simply re-run it, then re-run this repo-root detection cell.

If you need to fix it manually in Colab (new code cell):

```python
import subprocess
subprocess.run(["git", "-C", "/content/trading-agent", "fetch", "origin"], check=True)
subprocess.run(["git", "-C", "/content/trading-agent", "checkout", "major-rework"], check=True)
subprocess.run(["git", "-C", "/content/trading-agent", "pull", "origin", "major-rework"], check=True)
print("Done — now re-run from the repo-root detection cell")
```

**Fix B (no-GitHub upload):** run the **next** code cell once, upload the required `.py` files from your PC’s `feature_store/` folder, then **re-run** the repo-root detection cell.


In [ ]:
# Colab only: upload v43 modules if they are not on GitHub yet.
# Set RUN_V43_UPLOAD = True, run this cell, pick the two files from your PC, then re-run the repo-root cell.

RUN_V43_UPLOAD = False

from pathlib import Path

if not RUN_V43_UPLOAD:
    print("Skipping upload (set RUN_V43_UPLOAD = True to upload from PC).")
else:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("Upload path only works on Google Colab. Use git push on your PC.") from exc

    dest = Path("/content/trading-agent/feature_store")
    dest.mkdir(parents=True, exist_ok=True)
    print("Upload: jacksparrow_v43_build_matrix.py and jacksparrow_v43_contract.py")
    for name, data in files.upload().items():
        base = Path(name).name
        if not base.lower().endswith(".py"):
            continue
        out = dest / base
        out.write_bytes(data)
        print("Wrote", out, "(", out.stat().st_size, "bytes)")
    print("Done. Re-run the repo-root cell above, then continue.")


In [ ]:
import os
import sys
from pathlib import Path

# Colab: uncomment if your clone is not auto-detected
# os.environ["TRADING_AGENT_ROOT"] = "/content/trading-agent"

_MARKER = Path("feature_store") / "jacksparrow_v43_build_matrix.py"

def _repo_has_marker(c: Path) -> bool:
    try:
        return (c / _MARKER).is_file()
    except OSError:
        return False


candidates: list[Path] = []
if os.environ.get("TRADING_AGENT_ROOT"):
    candidates.append(Path(os.environ["TRADING_AGENT_ROOT"]).resolve())
# Default clone targets (see section 1). `git clone <url>` alone creates /content/<repo_name>/.
for fixed in ("/content/trading-agent", "/content/JackSparrow", "/content/jacksparrow"):
    candidates.append(Path(fixed).resolve())
candidates.append(Path.cwd().resolve())
p = Path.cwd().resolve()
for _ in range(16):
    candidates.append(p)
    if p == p.parent:
        break
    p = p.parent
_content = Path("/content")
if _content.is_dir():
    try:
        _skip_names = frozenset({"sample_data", ".config"})
        for child in sorted(_content.iterdir()):
            if child.is_dir() and not child.name.startswith(".") and child.name not in _skip_names:
                candidates.append(child.resolve())
    except OSError:
        pass

seen: set[Path] = set()
deduped: list[Path] = []
for c in candidates:
    try:
        r = c.resolve()
    except OSError:
        continue
    if r not in seen:
        seen.add(r)
        deduped.append(r)

REPO_ROOT = None
for c in deduped:
    if _repo_has_marker(c):
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    listing = ""
    if _content.is_dir():
        try:
            names = sorted(
                p.name for p in _content.iterdir() if p.is_dir() and not p.name.startswith(".")
            )
            listing = f" Top-level folders under /content: {names!r}."
        except OSError:
            pass
    print(f"Diagnosing: Checking for {_MARKER} in detected candidates.")
    for c in deduped:
        expected_path = c / _MARKER
        print(f"  Checking: {expected_path} (exists: {expected_path.exists()}, is_file: {expected_path.is_file()})")
    raise FileNotFoundError(
        f"Could not find repo root (need {_MARKER.as_posix()}). "
        "Push that file to GitHub (or merge branch that has it), then `git pull` under your clone, "
        "or set TRADING_AGENT_ROOT to a checkout that contains feature_store/. See notebook section 1b (git push or Colab upload)."
        + listing
    )

sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT =", REPO_ROOT)


In [ ]:
# Optional: verify v43 training modules exist on REPO_ROOT before heavy downloads
from pathlib import Path

_fs = Path(REPO_ROOT) / "feature_store"
for name in ("jacksparrow_v43_build_matrix.py", "jacksparrow_v43_contract.py"):
    p = _fs / name
    print(name, "OK" if p.is_file() else "MISSING — push from your PC then git pull in Colab")


In [ ]:
import importlib.metadata as importlib_metadata
import json
import os
import platform
import subprocess
import time
import warnings
from datetime import datetime, timezone

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import requests
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import RobustScaler
from xgboost import XGBRegressor

from feature_store.jacksparrow_v43_build_matrix import build_v43_feature_matrix
from feature_store.jacksparrow_v43_contract import (
    V43_CANONICAL_FEATURES,
    V43_COMPATIBLE_FEATURE_VERSION,
    V43_FORWARD_TARGET_BARS,
    validate_v43_metadata_compatibility,
)
from agent.models.v43_pickle_shims import (
    EnsembleModel,
    FeatureEngineer,
    LGBMModel,
    set_v43_build_feature_matrix,
)

warnings.filterwarnings("ignore", category=UserWarning)



## 2) Delta Exchange India — historical candles

Public `GET /v2/history/candles` on `https://api.india.delta.exchange` (same as
`agent/data/historical_data_loader.py`). Use `page_size` ≤ 2000 and polite delays between calls.


In [ ]:
DELTA_BASE = os.environ.get("DELTA_EXCHANGE_BASE_URL", "https://api.india.delta.exchange")
SYMBOL = os.environ.get("DELTA_SYMBOL", "BTCUSD")
RESOLUTION_5M = "5m"

TARGET_CANDLES_5M = int(os.environ.get("TARGET_CANDLES_5M", "60000"))
FUNDING_LOOKBACK_HOURS = int(os.environ.get("FUNDING_LOOKBACK_HOURS", "8000"))

REQUEST_DELAY_S = float(os.environ.get("DELTA_REQUEST_DELAY_S", "0.35"))
PAGE_SIZE = 2000

RESOLUTION_SECONDS = {"5m": 300, "15m": 900, "30m": 1800, "1h": 3600, "2h": 7200}


def _ts_col(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in ("open", "high", "low", "close", "volume"):
        out[c] = pd.to_numeric(out[c], errors="coerce")
    out["timestamp"] = pd.to_datetime(out["time"], unit="s", utc=True)
    out = out.drop(columns=["time"], errors="ignore")
    return out.sort_values("timestamp").reset_index(drop=True)


def fetch_candles_range(session, symbol, resolution, start_ts, end_ts, max_retries=4):
    url = f"{DELTA_BASE}/v2/history/candles"
    params = {
        "symbol": symbol,
        "resolution": resolution,
        "start": int(start_ts),
        "end": int(end_ts),
        "page_size": PAGE_SIZE,
    }
    last_exc: Exception = RuntimeError("no attempts made")
    for attempt in range(max_retries):
        try:
            r = session.get(url, params=params, timeout=30)
            r.raise_for_status()
            data = r.json()
            if not data.get("success"):
                raise RuntimeError(data)
            rows = data.get("result") or []
            return pd.DataFrame(rows)
        except (requests.exceptions.RequestException, RuntimeError) as exc:
            last_exc = exc
            if attempt < max_retries - 1:
                backoff = 2 ** attempt
                print(f"  fetch_candles_range retry {attempt + 1}/{max_retries - 1} in {backoff}s: {exc}")
                time.sleep(backoff)
    raise last_exc


def fetch_5m_history(symbol, target_rows, resolution="5m"):
    session = requests.Session()
    session.headers.update({"Accept": "application/json"})
    sec = RESOLUTION_SECONDS[resolution]
    end_ts = int(time.time())
    span = int(target_rows * sec * 1.15)
    start_ts = end_ts - span
    frames = []
    cursor = start_ts
    while cursor < end_ts:
        chunk_end = min(end_ts, cursor + PAGE_SIZE * sec)
        df = fetch_candles_range(session, symbol, resolution, cursor, chunk_end)
        if not df.empty:
            frames.append(df)
        cursor = chunk_end
        time.sleep(REQUEST_DELAY_S)
    if not frames:
        return pd.DataFrame()
    raw = pd.concat(frames, ignore_index=True)
    raw = raw.drop_duplicates(subset=["time"]).sort_values("time")
    if len(raw) > target_rows:
        raw = raw.iloc[-target_rows:]
    return _ts_col(raw)


def fetch_funding_hourly(symbol, lookback_hours):
    session = requests.Session()
    session.headers.update({"Accept": "application/json"})
    end_ts = int(time.time())
    start_ts = end_ts - int(lookback_hours * 3600)
    frames = []
    cursor = start_ts
    while cursor < end_ts:
        chunk_end = min(end_ts, cursor + PAGE_SIZE * 3600)
        df = fetch_candles_range(session, f"FUNDING:{symbol}", "1h", cursor, chunk_end)
        if not df.empty:
            frames.append(df)
        cursor = chunk_end
        time.sleep(REQUEST_DELAY_S)
    if not frames:
        return pd.DataFrame(columns=["timestamp", "funding_rate"])
    raw = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["time"]).sort_values("time")
    raw["timestamp"] = pd.to_datetime(raw["time"], unit="s", utc=True)
    raw["funding_rate"] = pd.to_numeric(raw["close"], errors="coerce")
    return raw[["timestamp", "funding_rate"]].reset_index(drop=True)


print("Fetching 5m OHLCV ...")
df_5m = fetch_5m_history(SYMBOL, TARGET_CANDLES_5M, RESOLUTION_5M)
print("5m rows:", len(df_5m), "range:", df_5m["timestamp"].min(), "->", df_5m["timestamp"].max())

print("Fetching funding (1h) ...")
try:
    df_funding = fetch_funding_hourly(SYMBOL, FUNDING_LOOKBACK_HOURS)
    print("funding rows:", len(df_funding))
except Exception as _fund_exc:
    print(f"WARNING: funding fetch failed ({_fund_exc!r}). Continuing with empty funding data.")
    df_funding = pd.DataFrame(columns=["timestamp", "funding_rate"])



## 3) Features and labels (v43 contract)

- Features: `build_v43_feature_matrix(df_5m, None, None, df_funding, for_training=True, primary_interval="5m")`
  then slice `V43_CANONICAL_FEATURES` in contract order.
- Label: **simple** forward return on 5m close:
  `close.shift(-V43_FORWARD_TARGET_BARS) / close - 1.0` aligned to the feature matrix index.


In [ ]:
df_feat = build_v43_feature_matrix(
    df_5m,
    None,
    None,
    df_funding,
    for_training=True,
    primary_interval="5m",
)

close = pd.to_numeric(df_5m["close"], errors="coerce")
y_raw = close.shift(-V43_FORWARD_TARGET_BARS) / close - 1.0
y_raw = y_raw.reindex(df_feat.index)

feat_cols = list(V43_CANONICAL_FEATURES)
X_all = df_feat[feat_cols].replace([np.inf, -np.inf], np.nan)
mask = X_all.notna().all(axis=1) & y_raw.notna() & np.isfinite(y_raw)
X = X_all.loc[mask].astype(np.float32)
y_series = y_raw.loc[mask].astype(np.float64)
y = y_series.values

if "timestamp" in df_feat.columns:
    timestamp_series = pd.to_datetime(df_feat.loc[X.index, "timestamp"], utc=True, errors="coerce")
else:
    timestamp_series = pd.Series(pd.NaT, index=X.index, dtype="datetime64[ns, UTC]")


def _ts_iso(ts):
    """Return ISO string for a Timestamp; return None for NaT or any failure."""
    try:
        return ts.isoformat()
    except Exception:
        return None


print("Training rows after dropna:", len(X))

VAL_FRAC = 0.15
EMBARGO_BARS = V43_FORWARD_TARGET_BARS
n = len(X)
val_start = int(n * (1.0 - VAL_FRAC))
train_end = max(0, val_start - EMBARGO_BARS)
if train_end <= 0 or val_start >= n:
    raise ValueError(
        f"Not enough rows for embargoed split: n={n}, val_start={val_start}, "
        f"embargo={EMBARGO_BARS}"
    )

X_train, X_val = X.iloc[:train_end], X.iloc[val_start:]
y_train, y_val = y[:train_end], y[val_start:]
embargo_index = X.index[train_end:val_start]

split_metadata = {
    "split_method": "chronological_embargo",
    "validation_fraction": VAL_FRAC,
    "embargo_bars": EMBARGO_BARS,
    "rows_total_after_dropna": int(n),
    "rows_train": int(len(X_train)),
    "rows_embargo": int(len(embargo_index)),
    "rows_validation": int(len(X_val)),
    "train_start": _ts_iso(timestamp_series.iloc[0]) if len(timestamp_series) else None,
    "train_end": _ts_iso(timestamp_series.loc[X_train.index[-1]]) if len(X_train) else None,
    "embargo_start": _ts_iso(timestamp_series.loc[embargo_index[0]]) if len(embargo_index) else None,
    "embargo_end": _ts_iso(timestamp_series.loc[embargo_index[-1]]) if len(embargo_index) else None,
    "validation_start": _ts_iso(timestamp_series.loc[X_val.index[0]]) if len(X_val) else None,
    "validation_end": _ts_iso(timestamp_series.loc[X_val.index[-1]]) if len(X_val) else None,
}

print("train:", len(X_train), "embargo:", len(embargo_index), "val:", len(X_val))
print("split_metadata:", json.dumps(split_metadata, indent=2))



## 4) Train `EnsembleModel` (LGBM + XGB + RF)

Matches `agent/models/v43_pickle_shims.EnsembleModel`: `_ens_scaler` for XGB/RF; `lgbm_model` is `LGBMModel`
with its own `scaler` + `lgbm` regressor. Before §4c, `EnsembleModel.predict` averages base regressors;
after §4c (Pattern 3), `predict` routes through **meta classifier + Ridge calibrator** (see §4c).

This cell records **regressor-baseline** validation metrics. §4c recomputes `dynamic_threshold` /
`short_threshold` and promotion metrics on the meta-stack path so exported thresholds match runtime inference.

`regime_thresholds` on `lgbm_model` use validation **P75** of the active inference path (see `JackSparrowV43Node._apply_v43_threshold_patch`).


In [ ]:
RNG = 42

lgbm_model = LGBMModel()
lgbm_model.scaler = RobustScaler()
X_train_lgb = lgbm_model.scaler.fit_transform(X_train.values)
X_val_lgb = lgbm_model.scaler.transform(X_val.values)

lgb_inner = lgb.LGBMRegressor(
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=8,
    min_child_samples=100,
    reg_lambda=1.0,
    reg_alpha=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RNG,
    verbose=-1,
)
lgb_inner.fit(X_train_lgb, y_train, eval_set=[(X_val_lgb, y_val)], eval_metric="l2")
lgbm_model.lgbm = lgb_inner
lgbm_model.feature_cols = feat_cols

ensemble = EnsembleModel()
ensemble._ens_scaler = RobustScaler()
X_train_s = ensemble._ens_scaler.fit_transform(X_train.values)
X_val_s = ensemble._ens_scaler.transform(X_val.values)

xgb = XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    min_child_weight=10,
    reg_lambda=1.0,
    reg_alpha=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RNG,
    n_jobs=-1,
    verbosity=0,
    early_stopping_rounds=30,
)
xgb.fit(X_train_s, y_train, eval_set=[(X_val_s, y_val)], verbose=False)

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=100,
    max_features=0.5,
    random_state=RNG,
    n_jobs=-1,
)
rf.fit(X_train_s, y_train)

ensemble.lgbm_model = lgbm_model
ensemble.xgb = xgb
ensemble.rf = rf
ensemble.feature_cols = feat_cols
ensemble._is_fitted = True

val_pred = np.asarray(ensemble.predict(X_val.values, X_df=X_val), dtype=np.float64).ravel()
dt_long = float(np.nanpercentile(val_pred, 75))
dt_short = float(np.nanpercentile(val_pred, 25))
dt_long = max(dt_long, 1e-6)
dt_short = min(dt_short, -1e-6)
dt_short_mag = abs(dt_short)

ensemble.threshold = dt_long
ensemble.dynamic_threshold = dt_long
ensemble.short_threshold = dt_short_mag
lgbm_model.dynamic_threshold = dt_long
lgbm_model.short_threshold = dt_short_mag
lgbm_model.regime_thresholds = {
    "neutral": dt_long,
    "ranging": dt_long,
    "trending": dt_long,
    "crisis": dt_long,
}


def _safe_corr(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    if len(a) < 2 or np.nanstd(a) == 0 or np.nanstd(b) == 0:
        return None
    return float(np.corrcoef(a, b)[0, 1])


def _max_drawdown(curve):
    curve = np.asarray(curve, dtype=np.float64)
    if curve.size == 0:
        return 0.0
    peak = np.maximum.accumulate(curve)
    return float(np.min(curve - peak))


def _candidate_stats(mask, gross_returns, net_returns):
    mask = np.asarray(mask, dtype=bool)
    gross = np.asarray(gross_returns, dtype=np.float64)[mask]
    net = np.asarray(net_returns, dtype=np.float64)[mask]
    if gross.size == 0:
        return {
            "count": 0,
            "coverage": 0.0,
            "gross_mean_return": None,
            "net_mean_return": None,
            "net_median_return": None,
            "hit_rate_gross_positive": None,
            "hit_rate_net_positive": None,
            "max_drawdown_proxy": 0.0,
        }
    return {
        "count": int(gross.size),
        "coverage": float(gross.size / max(1, len(mask))),
        "gross_mean_return": float(np.mean(gross)),
        "net_mean_return": float(np.mean(net)),
        "net_median_return": float(np.median(net)),
        "hit_rate_gross_positive": float(np.mean(gross > 0.0)),
        "hit_rate_net_positive": float(np.mean(net > 0.0)),
        "max_drawdown_proxy": _max_drawdown(np.cumsum(net)),
    }


maker_fee = float(os.environ.get("JACKSPARROW_V43_MAKER_FEE_RATE", "0.0005"))
slippage = float(os.environ.get("JACKSPARROW_V43_SLIPPAGE_PCT", "0.0003"))
leverage = int(os.environ.get("JACKSPARROW_V43_LEVERAGE_ASSUMPTION", "3"))
take_profit_pct = float(os.environ.get("JACKSPARROW_V43_TAKE_PROFIT_PCT", "0.015"))
min_edge_cost_ratio = float(os.environ.get("JACKSPARROW_V43_MIN_EDGE_COST_RATIO", "0.75"))
round_trip_cost = 2.0 * (maker_fee + slippage) * max(1, leverage)

val_rmse = float(np.sqrt(np.mean((val_pred - y_val) ** 2)))
val_mae = float(np.mean(np.abs(val_pred - y_val)))
directional_mask = y_val != 0
long_candidates = val_pred > dt_long
short_candidates = val_pred < dt_short
long_net = y_val - round_trip_cost
short_gross = -y_val
short_net = short_gross - round_trip_cost

validation_metrics = {
    "model_family": "jacksparrow_v43_regression",
    "target": "simple_forward_return",
    "inference_path": "regressor_mean",
    "threshold_source": "validation_prediction_percentile",
    "threshold_percentile": 75.0,
    "short_threshold_percentile": 25.0,
    "dynamic_threshold": dt_long,
    "short_threshold": dt_short_mag,
    "validation_rmse": val_rmse,
    "validation_mae": val_mae,
    "validation_corr": _safe_corr(val_pred, y_val),
    "directional_accuracy": float(
        np.mean(np.sign(val_pred[directional_mask]) == np.sign(y_val[directional_mask]))
    ) if np.any(directional_mask) else None,
    "prediction_mean": float(np.mean(val_pred)),
    "prediction_std": float(np.std(val_pred)),
    "target_mean": float(np.mean(y_val)),
    "target_std": float(np.std(y_val)),
    "runtime_cost_assumptions": {
        "maker_fee_rate": maker_fee,
        "slippage_pct": slippage,
        "leverage_assumption": leverage,
        "take_profit_pct": take_profit_pct,
        "min_edge_cost_ratio": min_edge_cost_ratio,
        "round_trip_cost_pct": round_trip_cost,
    },
    "long_candidates": _candidate_stats(long_candidates, y_val, long_net),
    "short_candidates": _candidate_stats(short_candidates, short_gross, short_net),
}

train_lgb_rmse = float(np.sqrt(np.mean((lgb_inner.predict(X_train_lgb) - y_train) ** 2)))
validation_metrics["train_lgb_inner_rmse"] = train_lgb_rmse
validation_metrics["regressor_baseline"] = {
    "inference_path": "regressor_mean",
    "dynamic_threshold": dt_long,
    "short_threshold": dt_short_mag,
    "validation_rmse": validation_metrics["validation_rmse"],
    "validation_mae": validation_metrics["validation_mae"],
    "validation_corr": validation_metrics["validation_corr"],
    "directional_accuracy": validation_metrics["directional_accuracy"],
    "long_candidates": validation_metrics["long_candidates"],
    "short_candidates": validation_metrics["short_candidates"],
}

print("regressor baseline dynamic_threshold (val P75):", dt_long)
print("regressor baseline short_threshold magnitude (val P25):", dt_short_mag)
print("train RMSE (LGB inner):", train_lgb_rmse)
print("regressor_baseline validation_metrics:")
print(json.dumps(validation_metrics["regressor_baseline"], indent=2))

if validation_metrics["validation_corr"] is not None and validation_metrics["validation_corr"] < 0:
    print(
        "WARNING: regressor validation_corr="
        f"{validation_metrics['validation_corr']:.4f} < 0 — §4c meta stack may still promote if meta corr > 0."
    )
if validation_metrics["directional_accuracy"] < 0.50:
    print("WARNING: regressor directional_accuracy < 50% — model barely separates direction.")



## 4c) Meta-learner stacking (Pattern 3)

Base regressors (LGBM + XGB + RF) are unchanged. A **LGBMClassifier** meta-learner maps the
stacked regressor outputs to direction probability; a **Ridge** calibrator maps probability
back to expected-return scale so gate-5 edge-vs-cost math stays valid at runtime.

Stored on ``ensemble.meta`` and ``ensemble.calibrator`` — routed automatically by
``EnsembleModel.predict()`` in ``v43_pickle_shims.py``.

After training, this cell **recomputes** ``dynamic_threshold`` / ``short_threshold`` from validation
**P75/P25 of meta-stack predictions** and refreshes ``validation_metrics`` for export (regressor baseline
is preserved under ``validation_metrics["regressor_baseline"]``).


In [ ]:
from lightgbm import LGBMClassifier
from sklearn.linear_model import Ridge
from sklearn.metrics import log_loss, roc_auc_score

X_stack_train = ensemble._base_predictions(X_train.values)
X_stack_val = ensemble._base_predictions(X_val.values)

y_bin_train = (y_train > 0).astype(int)
y_bin_val = (y_val > 0).astype(int)

meta_clf = LGBMClassifier(
    n_estimators=200,
    num_leaves=15,
    min_child_samples=100,
    reg_lambda=2.0,
    reg_alpha=0.1,
    learning_rate=0.05,
    random_state=RNG,
    verbose=-1,
)
meta_clf.fit(
    X_stack_train,
    y_bin_train,
    eval_set=[(X_stack_val, y_bin_val)],
    callbacks=[lgb.early_stopping(20, verbose=False)],
)

meta_val_proba = meta_clf.predict_proba(X_stack_val)[:, 1]
calibrator = Ridge(alpha=1.0)
calibrator.fit(meta_val_proba.reshape(-1, 1), y_val)

ensemble.meta = meta_clf
ensemble.calibrator = calibrator

meta_log_loss = log_loss(y_bin_val, meta_val_proba)
meta_auc = roc_auc_score(y_bin_val, meta_val_proba)
print(f"meta val log-loss={meta_log_loss:.4f}, AUC={meta_auc:.4f}")
print("calibrator coef:", calibrator.coef_, "intercept:", calibrator.intercept_)

if meta_auc < 0.50:
    raise RuntimeError(
        f"Meta classifier AUC={meta_auc:.4f} < 0.50. Worse than random — retune."
    )

val_pred_meta = np.asarray(
    ensemble.predict(X_val.values, X_df=X_val), dtype=np.float64
).ravel()

# Recompute thresholds on meta-stack predictions (matches runtime inference path).
dt_long = max(float(np.nanpercentile(val_pred_meta, 75)), 1e-6)
dt_short = min(float(np.nanpercentile(val_pred_meta, 25)), -1e-6)
dt_short_mag = abs(dt_short)
ensemble.threshold = dt_long
ensemble.dynamic_threshold = dt_long
ensemble.short_threshold = dt_short_mag
lgbm_model.dynamic_threshold = dt_long
lgbm_model.short_threshold = dt_short_mag
lgbm_model.regime_thresholds = {
    "neutral": dt_long,
    "ranging": dt_long,
    "trending": dt_long,
    "crisis": dt_long,
}

long_candidates_meta = val_pred_meta > dt_long
short_candidates_meta = val_pred_meta < dt_short
validation_corr_meta = _safe_corr(val_pred_meta, y_val)

validation_metrics["inference_path"] = "meta_calibrator"
validation_metrics["threshold_source"] = "validation_meta_prediction_percentile"
validation_metrics["dynamic_threshold"] = dt_long
validation_metrics["short_threshold"] = dt_short_mag
validation_metrics["validation_rmse"] = float(np.sqrt(np.mean((val_pred_meta - y_val) ** 2)))
validation_metrics["validation_mae"] = float(np.mean(np.abs(val_pred_meta - y_val)))
validation_metrics["validation_corr"] = validation_corr_meta
validation_metrics["validation_corr_meta"] = validation_corr_meta
validation_metrics["directional_accuracy"] = float(
    np.mean(np.sign(val_pred_meta[directional_mask]) == np.sign(y_val[directional_mask]))
) if np.any(directional_mask) else None
validation_metrics["prediction_mean"] = float(np.mean(val_pred_meta))
validation_metrics["prediction_std"] = float(np.std(val_pred_meta))
validation_metrics["long_candidates"] = _candidate_stats(long_candidates_meta, y_val, long_net)
validation_metrics["short_candidates"] = _candidate_stats(short_candidates_meta, short_gross, short_net)
validation_metrics["meta_auc"] = float(meta_auc)
validation_metrics["meta_log_loss"] = float(meta_log_loss)
validation_metrics["prediction_mean_meta"] = float(np.mean(val_pred_meta))

print("meta-stack dynamic_threshold (val P75):", dt_long)
print("meta-stack short_threshold magnitude (val P25):", dt_short_mag)
print("meta-stack validation_corr:", validation_corr_meta)
print("meta-stack val prediction_mean:", validation_metrics["prediction_mean_meta"])
print("promotion validation_metrics (meta path):")
print(json.dumps(validation_metrics, indent=2))

if validation_corr_meta is not None and validation_corr_meta < 0:
    raise RuntimeError(
        f"Export blocked: validation_corr_meta={validation_corr_meta:.4f} < 0. "
        "Meta-stack predictions are inversely correlated with returns. Retune."
    )
if validation_metrics["directional_accuracy"] is not None and validation_metrics["directional_accuracy"] < 0.50:
    print("WARNING: meta-stack directional_accuracy < 50%.")


In [ ]:
## 4b) Walk-forward threshold stability analysis
# Splits the full dataset into WF_WINDOWS rolling windows of equal size and evaluates
# the trained ensemble on each. A CoV (std/mean) of P75 threshold < 0.5 indicates the
# OOF-derived dynamic_threshold is stable enough to trust across market regimes.
# Note: this is a same-model evaluation, not independent retraining per fold.

WF_WINDOWS = 10
wf_step = max(50, n // WF_WINDOWS)
wf_results = []

def _wf_ts_iso(i):
    if i < 0 or i >= len(timestamp_series):
        return "?"
    try:
        return timestamp_series.iloc[i].isoformat()
    except Exception:
        return "?"

for w_start in range(0, n, wf_step):
    w_end = min(w_start + wf_step, n)
    Xw_df = X.iloc[w_start:w_end]
    yw = y[w_start:w_end]
    if len(Xw_df) < 50:
        continue
    pred_w = np.asarray(ensemble.predict(Xw_df.values, X_df=Xw_df), dtype=np.float64).ravel()
    rmse_w = float(np.sqrt(np.mean((pred_w - yw) ** 2)))
    corr_w = _safe_corr(pred_w, yw)
    dir_mask = yw != 0
    dir_acc_w = float(np.mean(np.sign(pred_w[dir_mask]) == np.sign(yw[dir_mask]))) if np.any(dir_mask) else None
    p75_w = float(np.nanpercentile(pred_w, 75))

    ts_s = _wf_ts_iso(w_start)
    ts_e = _wf_ts_iso(w_end - 1)
    wf_results.append({
        "window": len(wf_results) + 1,
        "ts_start": ts_s[:10],
        "ts_end": ts_e[:10],
        "rows": int(w_end - w_start),
        "rmse": rmse_w,
        "corr": corr_w,
        "directional_acc": dir_acc_w,
        "p75_threshold": p75_w,
    })
    corr_str = f"{corr_w:.3f}" if corr_w is not None else "n/a"
    dir_str = f"{dir_acc_w:.3f}" if dir_acc_w is not None else "n/a"
    print(
        f"W{len(wf_results):02d} {ts_s[:10]}→{ts_e[:10]} "
        f"| rmse={rmse_w:.6f} corr={corr_str} dir={dir_str} p75={p75_w:.6f}"
    )

if not wf_results:
    print("No walk-forward windows produced (dataset may be too small after filters).")
else:
    thr_vals = [r["p75_threshold"] for r in wf_results]
    mean_thr = float(np.mean(thr_vals))
    std_thr = float(np.std(thr_vals))
    cov_thr = std_thr / max(abs(mean_thr), 1e-9)
    print(f"\nP75 threshold stability across {len(wf_results)} windows (CoV target < 0.5):")
    print(f"  mean={mean_thr:.6f}  std={std_thr:.6f}  CoV={cov_thr:.3f}")
    print(f"  range=[{min(thr_vals):.6f}, {max(thr_vals):.6f}]")
    if cov_thr >= 0.5:
        print("  WARNING: high threshold variance — consider a more conservative dynamic_threshold.")
    else:
        print("  OK: threshold is stable across market windows.")
    print(json.dumps(wf_results, indent=2))


## 5) `FeatureEngineer` + export

`FeatureEngineer` from `v43_pickle_shims` stores only the canonical column contract. The actual transform path is registered with `set_v43_build_feature_matrix(...)`, matching the agent startup path and avoiding fragile notebook-local closure pickling.


In [ ]:
def make_feature_engineer():
    fe = FeatureEngineer()
    fe.columns = list(V43_CANONICAL_FEATURES)
    return fe


def _version_or_none(package_name):
    try:
        return importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        return None


def _git_commit_or_none():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            cwd=str(REPO_ROOT),
            stderr=subprocess.DEVNULL,
            text=True,
        ).strip()
    except Exception:
        return None


set_v43_build_feature_matrix(build_v43_feature_matrix)
fe = make_feature_engineer()

OUT_DIR = Path(os.environ.get("V43_EXPORT_DIR", "jacksparrow_v43_bundle"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
artifact_path = OUT_DIR / "model_artifact_v43.pkl"
metadata_path = OUT_DIR / "metadata_v43.json"

assert validation_metrics.get("inference_path") == "meta_calibrator", (
    "Run §4c before export so thresholds and metrics match the meta-stack inference path."
)
assert validation_metrics["validation_corr"] > 0, "Export blocked: negative meta-stack correlation"
assert validation_metrics.get("meta_auc", 0) > 0.50, "Export blocked: meta classifier AUC <= 0.50"

# Hit-rate gates: gross >50% is desirable but not a hard block at low-AUC signal levels.
# Model is viable as long as corr > 0 and meta_auc > 0.50; hit-rate warnings are informational.
_long_hr = validation_metrics["long_candidates"].get("hit_rate_gross_positive") or 0.0
_short_hr = validation_metrics["short_candidates"].get("hit_rate_gross_positive") or 0.0
if _long_hr < 0.50:
    print(f"WARNING: long hit rate (gross) = {_long_hr:.4f} < 0.50 — "
          "model is directionally weak on long candidates; review before live deployment.")
if _short_hr < 0.50:
    print(f"WARNING: short hit rate (gross) = {_short_hr:.4f} < 0.50 — "
          "model is directionally weak on short candidates; review before live deployment.")

artifact_payload = {
    "model": ensemble,
    "feature_engineer": fe,
    "features": list(V43_CANONICAL_FEATURES),
    "validation_metrics": validation_metrics,
}
joblib.dump(artifact_payload, artifact_path)

_agent_load_snippet = "\n".join(
    [
        "artifact = joblib.load(MODEL_ARTIFACT_PATH)",
        "model    = artifact[\"model\"]",
        "features = artifact[\"features\"]",
        "fe       = artifact[\"feature_engineer\"]",
        "df_feat  = fe.transform(df5, df15, df1h, df_funding, include_target=False)",
        "assert len(df_feat) >= 2, \"Need at least 2 rows to use the last closed bar\"",
        "X        = df_feat[features].iloc[[-2]].values",
        "X_df     = df_feat[features].iloc[[-2]]",
        "expected_return = model.predict(X, X_df=X_df)",
    ]
)

meta = {
    "version": "v43",
    "compatible_feature_version": V43_COMPATIBLE_FEATURE_VERSION,
    "symbol": SYMBOL,
    "model_name": "jacksparrow_v43_" + str(SYMBOL),
    "exported_at": datetime.now(timezone.utc).isoformat(),
    "feature_count": len(V43_CANONICAL_FEATURES),
    "features": list(V43_CANONICAL_FEATURES),
    "training_forward_bars": V43_FORWARD_TARGET_BARS,
    "target_definition": "simple_forward_return",
    "model_architecture": {
        "base_estimators": ["lgbm_regressor", "xgb_regressor", "rf_regressor"],
        "meta_learner": "lgbm_classifier" if ensemble.meta is not None else None,
        "calibrator": "ridge" if ensemble.calibrator is not None else None,
    },
    "split": split_metadata,
    "validation_metrics": validation_metrics,
    "data": {
        "delta_base_url": DELTA_BASE,
        "symbol": SYMBOL,
        "resolution": RESOLUTION_5M,
        "target_candles_5m": TARGET_CANDLES_5M,
        "funding_lookback_hours": FUNDING_LOOKBACK_HOURS,
        "candles_5m_rows": int(len(df_5m)),
        "funding_rows": int(len(df_funding)),
        "candles_5m_start": df_5m["timestamp"].min().isoformat() if len(df_5m) else None,
        "candles_5m_end": df_5m["timestamp"].max().isoformat() if len(df_5m) else None,
    },
    "provenance": {
        "repo_root": str(REPO_ROOT),
        "git_commit": _git_commit_or_none(),
        "python": platform.python_version(),
        "platform": platform.platform(),
        "packages": {
            "joblib": _version_or_none("joblib"),
            "numpy": _version_or_none("numpy"),
            "pandas": _version_or_none("pandas"),
            "scikit-learn": _version_or_none("scikit-learn"),
            "lightgbm": _version_or_none("lightgbm"),
            "xgboost": _version_or_none("xgboost"),
        },
    },
    "agent_load": _agent_load_snippet,
}
metadata_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
print("Wrote", artifact_path)
print("Wrote", metadata_path)



## 6) Alignment self-check + handoff

Run `validate_v43_metadata_compatibility` on the exported JSON. Then copy the bundle folder into
`agent/model_storage/` (for example `JackSparrow_v43_models_BTCUSD/`) and configure model discovery per
`docs/03-ml-models.md`.

**On your machine (after download):**

```text
pytest tests/unit/test_jacksparrow_v43_contract.py
python scripts/smoke_test_v43.py
pytest tests/unit/test_jack_sparrow_v43_node_ctx.py
```


In [ ]:
with metadata_path.open(encoding="utf-8") as f:
    meta_loaded = json.load(f)
validate_v43_metadata_compatibility(meta_loaded)

set_v43_build_feature_matrix(build_v43_feature_matrix)
tail5 = df_5m.tail(800).copy()
sm = fe.transform(tail5, None, None, df_funding, include_target=False)
assert len(sm) >= 2
assert list(sm.columns[: len(V43_CANONICAL_FEATURES)]) == list(V43_CANONICAL_FEATURES)
expected_return = ensemble.predict(
    sm[V43_CANONICAL_FEATURES].iloc[[-2]].values,
    X_df=sm[V43_CANONICAL_FEATURES].iloc[[-2]],
)
assert np.atleast_1d(np.asarray(expected_return, dtype=np.float64)).size == 1
assert "validation_metrics" in meta_loaded and "split" in meta_loaded
assert meta_loaded["split"]["embargo_bars"] == V43_FORWARD_TARGET_BARS
print("validate_v43_metadata_compatibility: OK")
print("metadata split + validation metrics: OK")
print("fe.transform + ensemble.predict smoke: OK", expected_return)



In [ ]:
try:
    from google.colab import files
    import shutil

    z = Path("/content/jacksparrow_v43_bundle.zip")
    shutil.make_archive(str(z.with_suffix("")), "zip", root_dir=str(OUT_DIR))
    files.download(str(z))
except Exception as e:
    print("Colab zip/download skipped:", e)

